# Biomedical Inventory Reporting

## Business context
A new client report was requested, but the biomedical management team needed help creating its structure from the existing inventory. The source inventory was not defective and its structure was not to be changed.

This notebook adapts the existing inventory into a consolidated Excel workbook. The public demonstration uses fictional units, brands, models, and asset records.

## Counting rule
**One valid source row = one asset.** A quantity column is deliberately not used. Identical-looking rows are not deduplicated because separate assets may have identical descriptions, brands, and models.

## Run instructions
Install `pandas` and `openpyxl`, then run all cells in order. Demo mode is on by default and creates fictional input. For your own Excel, set `DEMO_MODE = False`; upload one file in Colab, or set `INPUT_PATH` in local Jupyter.

## Output
A consolidated sheet, a detail sheet for each functional unit, and a count reconciliation. Compatibility fields remain blank when source information is unavailable. Useful life is not calculated by this V2 adaptation.


## 1. Dependencies and configuration

Demo mode keeps the example independent of company data.

In [ ]:
# ============================================================
# 1. LIBRERÍAS
# ============================================================

import os
import re
import unicodedata
import warnings
from collections import Counter

import pandas as pd

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

try:
    from google.colab import files
except ImportError:
    files = None
from IPython.display import display

DEMO_MODE = True
INPUT_PATH = "input_inventory.xlsx"

SALIDA = "demo_inventory_report.xlsx"

print("Librerías cargadas.")

## 2. Fictional sample or single-file input

The source structure is preserved; the example follows the expected generic headers.

In [ ]:
# Synthetic input is generated in memory. No original inventory is included.
if DEMO_MODE:
    sample = pd.DataFrame([
        {
            "UNIDAD FUNCIONAL": f"Unit {1 + i % 3}",
            "ESPACIO FUNCIONAL": f"Area {1 + i % 2}",
            "DESCRIPCION": "Sample device" if i < 8 else "Sample furniture",
            "MARCA": "DemoBrand A" if i % 2 == 0 else "DemoBrand B",
            "MODELO": f"DemoModel {1 + i % 2}",
            "CLASIFICACION": "EQUIPO MEDICO" if i < 8 else "MOBILIARIO",
        }
        for i in range(12)
    ])
    demo_path = "synthetic_inventory.xlsx"
    sample.to_excel(demo_path, sheet_name="Consolidado", index=False)
    rutas_excel = [demo_path]
elif files is not None:
    subidos = files.upload()
    rutas_excel = [name for name in subidos if name.lower().endswith((".xlsx", ".xlsm"))]
    if len(rutas_excel) != 1:
        raise ValueError("Upload exactly one .xlsx or .xlsm inventory.")
else:
    if not os.path.isfile(INPUT_PATH):
        raise FileNotFoundError(INPUT_PATH)
    rutas_excel = [INPUT_PATH]
print("Input:", rutas_excel[0])

## 3. Header detection and text keys

Normalization is used for grouping keys and does not edit the original workbook.

In [ ]:
# ============================================================
# 3. FUNCIONES DE LIMPIEZA Y DETECCIÓN
# ============================================================

def texto_limpio(valor):
    if pd.isna(valor):
        return ""

    if isinstance(valor, float) and valor.is_integer():
        valor = int(valor)

    s = str(valor).replace("\xa0", " ")
    s = re.sub(r"\s+", " ", s).strip()

    if s.lower() in {"nan", "none"}:
        return ""

    return s


def normalizar(valor):
    s = texto_limpio(valor).upper()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(
        c for c in s
        if not unicodedata.combining(c)
    )
    s = re.sub(r"\s+", " ", s).strip()
    return s


def normalizar_modelo(valor):
    s = normalizar(valor)

    equivalentes_sin_modelo = {
        "",
        "NO APLICA",
        "N/A",
        "NA",
        "S/M",
        "SM",
        "SIN MODELO"
    }

    if s in equivalentes_sin_modelo:
        return "S/M"

    return s


def moda_no_vacia(serie):
    valores = [
        texto_limpio(x)
        for x in serie
        if texto_limpio(x)
    ]

    if not valores:
        return ""

    return Counter(valores).most_common(1)[0][0]


def nombre_hoja_seguro(nombre):
    nombre = texto_limpio(nombre)

    for c in ["\\", "/", "*", "?", ":", "[", "]"]:
        nombre = nombre.replace(c, " ")

    nombre = re.sub(r"\s+", " ", nombre).strip()

    if not nombre:
        nombre = "UF"

    return nombre[:31]


# ------------------------------------------------------------
# Alias de columnas permitidos
# ------------------------------------------------------------

ALIASES = {
    "UF": [
        "UNIDAD FUNCIONAL",
        "UF"
    ],
    "EF": [
        "ESPACIO FUNCIONAL",
        "LOCAL"
    ],
    "DESC": [
        "DESCRIPCION",
        "DESCRIPCIÓN",
        "DESCRIPCION DEL BIEN",
        "DESCRIPCIÓN DEL BIEN",
        "EQUIPO"
    ],
    "MARCA": [
        "MARCA"
    ],
    "MODELO": [
        "MODELO"
    ],
    "CLAS": [
        "CLASIFICACION",
        "CLASIFICACIÓN"
    ]
}


def buscar_posicion(valores, aliases):
    valores = [normalizar(x) for x in valores]

    for alias in aliases:
        alias_norm = normalizar(alias)

        for i, valor in enumerate(valores):
            if valor == alias_norm:
                return i

    return None


def detectar_inventario(ruta):
    try:
        xls = pd.ExcelFile(ruta)
    except Exception:
        return None

    candidatos = []

    for hoja in xls.sheet_names:

        try:
            preview = pd.read_excel(
                ruta,
                sheet_name=hoja,
                header=None,
                nrows=20,
                dtype=object
            )
        except Exception:
            continue

        for fila in range(len(preview)):

            valores = preview.iloc[fila].tolist()

            posiciones = {
                clave: buscar_posicion(valores, aliases)
                for clave, aliases in ALIASES.items()
            }

            obligatorias = [
                "UF",
                "EF",
                "DESC",
                "MARCA",
                "MODELO"
            ]

            if all(
                posiciones[c] is not None
                for c in obligatorias
            ):

                score = 100

                nombre_hoja = normalizar(hoja)

                if nombre_hoja in {
                    "CONDENSADO",
                    "CONSOLIDADO",
                    "CONCENTRADO"
                }:
                    score += 100

                candidatos.append({
                    "ruta": ruta,
                    "hoja": hoja,
                    "fila_header": fila,
                    "posiciones": posiciones,
                    "score": score
                })

    if not candidatos:
        return None

    ranked = sorted(candidatos, key=lambda x: x["score"], reverse=True)
    if len(ranked) > 1 and ranked[0]["score"] == ranked[1]["score"]:
        raise ValueError("Ambiguous inventory sheets or headers. Select one source table.")
    return ranked[0]


# ------------------------------------------------------------
# Detectar cuál archivo subido es el inventario.
# Normalmente será el único.
# ------------------------------------------------------------

detectados = []

for ruta in rutas_excel:
    info = detectar_inventario(ruta)

    if info:
        detectados.append(info)

if not detectados:
    raise ValueError(
        "No encontré en el archivo una hoja con las columnas necesarias: "
        "Unidad Funcional, Espacio Funcional, Descripción, Marca y Modelo."
    )

info_inv = sorted(
    detectados,
    key=lambda x: x["score"],
    reverse=True
)[0]

RUTA_INVENTARIO = info_inv["ruta"]

print("Inventario detectado correctamente:")
print("Archivo:", os.path.basename(RUTA_INVENTARIO))
print("Hoja   :", info_inv["hoja"])
print("Header :", info_inv["fila_header"] + 1)

## 4. Read the source without changing it

Rows with blank unit or description, and TOTAL rows, are excluded by the V2 counting rule.

In [ ]:
# ============================================================
# 4. LEER INVENTARIO
#    IMPORTANTE: CADA FILA = 1 PIEZA
# ============================================================

df_raw = pd.read_excel(
    RUTA_INVENTARIO,
    sheet_name=info_inv["hoja"],
    header=info_inv["fila_header"],
    dtype=object
)

columnas = list(df_raw.columns)
pos = info_inv["posiciones"]

col_uf = columnas[pos["UF"]]
col_ef = columnas[pos["EF"]]
col_desc = columnas[pos["DESC"]]
col_marca = columnas[pos["MARCA"]]
col_modelo = columnas[pos["MODELO"]]

col_clas = (
    columnas[pos["CLAS"]]
    if pos["CLAS"] is not None
    else None
)


df = pd.DataFrame({
    "UF": df_raw[col_uf].apply(texto_limpio),
    "EF": df_raw[col_ef].apply(texto_limpio),
    "DESCRIPCION": df_raw[col_desc].apply(texto_limpio),
    "MARCA": df_raw[col_marca].apply(texto_limpio),
    "MODELO": df_raw[col_modelo].apply(texto_limpio),
})

if col_clas:
    df["CLASIFICACION_ORIGEN"] = (
        df_raw[col_clas]
        .apply(texto_limpio)
    )
else:
    df["CLASIFICACION_ORIGEN"] = ""


# ============================================================
# REGLA DE CONTEO
# ============================================================
# Una fila = una pieza.
# NO se lee ni se utiliza ninguna columna de cantidad.

df["PIEZA"] = 1


# ------------------------------------------------------------
# Eliminar únicamente filas que no representan bienes válidos.
# ------------------------------------------------------------

df = df[
    df["UF"].ne("")
    & df["DESCRIPCION"].ne("")
].copy()

# Evitar posibles filas de totales dentro de la base.
df = df[
    df["UF"].apply(normalizar).ne("TOTAL")
].copy()


# ------------------------------------------------------------
# Claves normalizadas
# ------------------------------------------------------------

UF_ALIAS = {}  # No organization-specific aliases in the public version.


def canonizar_uf(valor):
    k = normalizar(valor)
    return UF_ALIAS.get(k, k)


df["UF_KEY"] = df["UF"].apply(canonizar_uf)
df["EF_KEY"] = df["EF"].apply(normalizar)
df["DESC_KEY"] = df["DESCRIPCION"].apply(normalizar)
df["MARCA_KEY"] = df["MARCA"].apply(normalizar_modelo)
df["MODELO_KEY"] = df["MODELO"].apply(normalizar_modelo)


print("Registros válidos :", len(df))
print("Piezas contadas   :", int(df["PIEZA"].sum()))

assert len(df) == int(df["PIEZA"].sum())

print("\n✓ Cada registro está contando exactamente como una pieza.")

## 5. Count each asset combination

Different brands or models remain separate; they are not assumed to be errors.

In [ ]:
# ============================================================
# 5. VALIDAR MARCA Y MODELO + HACER CONTEO
# ============================================================

# ------------------------------------------------------------
# Advertencia si la misma descripción aparece con
# distintas marcas o modelos.
# ------------------------------------------------------------

revision = (
    df.groupby("DESC_KEY")
      .agg(
          DESCRIPCION=("DESCRIPCION", moda_no_vacia),
          MARCAS=("MARCA_KEY", "nunique"),
          MODELOS=("MODELO_KEY", "nunique")
      )
      .reset_index()
)

inconsistencias = revision[
    (revision["MARCAS"] > 1)
    | (revision["MODELOS"] > 1)
].copy()

if inconsistencias.empty:
    print(
        "✓ Cada descripción conserva una sola combinación "
        "de Marca / Modelo."
    )
else:
    print(
        "AVISO: se encontraron descripciones con más de una "
        "Marca o Modelo."
    )
    print(
        "No se mezclarán: cada combinación se contará por separado."
    )

    display(
        inconsistencias[
            ["DESCRIPCION", "MARCAS", "MODELOS"]
        ].head(50)
    )


# ------------------------------------------------------------
# CONTEO
# ------------------------------------------------------------

detalle = (
    df.groupby(
        [
            "UF_KEY",
            "EF_KEY",
            "DESC_KEY",
            "MARCA_KEY",
            "MODELO_KEY"
        ],
        dropna=False
    )
    .agg(
        UF=("UF", moda_no_vacia),
        LOCAL=("EF", moda_no_vacia),
        EQUIPO=("DESCRIPCION", moda_no_vacia),
        MARCA=("MARCA", moda_no_vacia),
        MODELO=("MODELO", moda_no_vacia),
        CLASIFICACION_ORIGEN=(
            "CLASIFICACION_ORIGEN",
            moda_no_vacia
        ),
        CANTIDAD=("PIEZA", "sum")
    )
    .reset_index()
)

detalle["CANTIDAD"] = detalle["CANTIDAD"].astype(int)


# Clasificación compatible con el formato anterior.
MAPA_CLASIFICACION = {
    "MOBILIARIO": "M",
    "EQUIPO MEDICO": "EM",
    "EQUIPO INDUSTRIAL": "EI",
    "INSTRUMENTAL": "INS",
}

detalle["CLASIFICACION"] = (
    detalle["CLASIFICACION_ORIGEN"]
    .apply(
        lambda x: MAPA_CLASIFICACION.get(
            normalizar(x),
            texto_limpio(x)
        )
    )
)


print("\nPrimeros resultados del conteo:")
display(
    detalle[
        [
            "UF",
            "LOCAL",
            "EQUIPO",
            "CANTIDAD",
            "MARCA",
            "MODELO",
            "CLASIFICACION"
        ]
    ].head(15)
)

## 6. Determine output units

No actual facility unit map is published.

In [ ]:
# ============================================================
# 6. CONFIGURACIÓN DE UNIDADES FUNCIONALES
# ============================================================

# Orden y nombres siguiendo el formato utilizado anteriormente.
# Si en un inventario futuro aparece una UF nueva, el notebook
# también la agregará automáticamente.

UF_CONFIG = []  # Units are derived from the input, not a real facility map.

config_por_key = {
    key: {
        "key": key,
        "nombre_concentrado": nombre_concentrado,
        "nombre_hoja": nombre_hoja
    }
    for key, nombre_concentrado, nombre_hoja in UF_CONFIG
}


ufs_presentes = set(detalle["UF_KEY"].unique())

# Sólo se crearán pestañas para UFs presentes en este inventario.
configs_salida = []

for key, _, _ in UF_CONFIG:
    if key in ufs_presentes:
        configs_salida.append(config_por_key[key])


# UFs no contempladas previamente.
for uf_key in sorted(
    ufs_presentes - set(config_por_key)
):
    nombre_real = moda_no_vacia(
        detalle.loc[
            detalle["UF_KEY"] == uf_key,
            "UF"
        ]
    )

    configs_salida.append({
        "key": uf_key,
        "nombre_concentrado": nombre_real or uf_key,
        "nombre_hoja": nombre_hoja_seguro(nombre_real or uf_key)
    })


print("Unidades funcionales encontradas:", len(configs_salida))

for cfg in configs_salida:
    print(" -", cfg["nombre_concentrado"])

## 7. Build the Excel workbook

Output preserves the consolidation and unit-detail logic. Unknown compatibility values stay blank.

In [ ]:
# ============================================================
# 7. GENERAR EXCEL
# ============================================================

# ------------------------------------------------------------
# Estilos
# ------------------------------------------------------------

AZUL = "1F4E78"
BLANCO = "FFFFFF"
GRIS = "D9E1F2"

fill_header = PatternFill(
    fill_type="solid",
    fgColor=AZUL
)

fill_total = PatternFill(
    fill_type="solid",
    fgColor=GRIS
)

font_header = Font(
    bold=True,
    color=BLANCO,
    size=10
)

font_bold = Font(
    bold=True
)

lado_fino = Side(
    style="thin",
    color="B7B7B7"
)

border = Border(
    left=lado_fino,
    right=lado_fino,
    top=lado_fino,
    bottom=lado_fino
)


def aplicar_bordes(ws, min_row, max_row, min_col, max_col):
    # CORRECCIÓN:
    # openpyxl NO tiene cell.vertical_alignment.
    # Aquí sólo aplicamos el borde.
    # La alineación se establece correctamente con cell.alignment
    # donde sea necesaria.
    if max_row < min_row:
        return

    for row in ws.iter_rows(
        min_row=min_row,
        max_row=max_row,
        min_col=min_col,
        max_col=max_col
    ):
        for cell in row:
            cell.border = border


def ajustar_columnas(ws):
    for col_cells in ws.columns:

        letra = get_column_letter(
            col_cells[0].column
        )

        max_len = 0

        for cell in col_cells:
            if cell.value is not None:
                max_len = max(
                    max_len,
                    len(str(cell.value))
                )

        ws.column_dimensions[letra].width = min(
            max(max_len + 2, 10),
            45
        )


# ------------------------------------------------------------
# Crear libro
# ------------------------------------------------------------

wb = Workbook()
wb.remove(wb.active)


# ============================================================
# 7A. HOJA CONCENTRADO
# ============================================================

ws = wb.create_sheet("Concentrado")

headers = (
    [
        "No.",
        "DESCRIPCIÓN",
        "MARCA",
        "MODELO"
    ]
    +
    [
        cfg["nombre_concentrado"]
        for cfg in configs_salida
    ]
    +
    [
        "TOTAL DE EQUIPOS",
        "INSTALL_A",
        "INSTALL_B",
        "INSTALL_C",
        "M",
        "INSTALL_E",
        "CLASIFICACION",
        "VIDA UTIL (AÑOS)"
    ]
)

# No existe la columna ENCONTRADOS FÍSICAMENTE.
ws.append(headers)


# ------------------------------------------------------------
# Bienes únicos
# ------------------------------------------------------------

base_bienes = (
    df.groupby(
        [
            "DESC_KEY",
            "MARCA_KEY",
            "MODELO_KEY"
        ],
        dropna=False
    )
    .agg(
        DESCRIPCION=(
            "DESCRIPCION",
            moda_no_vacia
        ),
        MARCA=(
            "MARCA",
            moda_no_vacia
        ),
        MODELO=(
            "MODELO",
            moda_no_vacia
        ),
        CLASIFICACION_ORIGEN=(
            "CLASIFICACION_ORIGEN",
            moda_no_vacia
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# Matriz de conteos por UF
# ------------------------------------------------------------

pivot = (
    df.groupby(
        [
            "DESC_KEY",
            "MARCA_KEY",
            "MODELO_KEY",
            "UF_KEY"
        ],
        dropna=False
    )["PIEZA"]
    .sum()
    .unstack(fill_value=0)
)


base_bienes = (
    base_bienes
    .set_index(
        [
            "DESC_KEY",
            "MARCA_KEY",
            "MODELO_KEY"
        ]
    )
    .join(
        pivot,
        how="left"
    )
    .reset_index()
)


base_bienes = base_bienes.sort_values(
    by=[
        "DESCRIPCION",
        "MARCA",
        "MODELO"
    ],
    key=lambda s:
        s.astype(str).str.upper()
).reset_index(drop=True)


# ------------------------------------------------------------
# Escribir Concentrado
# ------------------------------------------------------------

for i, r in base_bienes.iterrows():

    cantidades = [
        int(
            r.get(
                cfg["key"],
                0
            ) or 0
        )
        for cfg in configs_salida
    ]

    total = sum(cantidades)

    clasificacion = (
        MAPA_CLASIFICACION.get(
            normalizar(
                r["CLASIFICACION_ORIGEN"]
            ),
            texto_limpio(
                r["CLASIFICACION_ORIGEN"]
            )
        )
    )

    ws.append([
        i + 1,
        r["DESCRIPCION"],
        r["MARCA"],
        r["MODELO"],
        *cantidades,
        total,

        # No hay Formato.xlsx:
        # estos campos permanecen vacíos.
        "",
        "",
        "",
        "",
        "",

        clasificacion,
        ""
    ])


# ------------------------------------------------------------
# Formato Concentrado
# ------------------------------------------------------------

for cell in ws[1]:

    cell.fill = fill_header
    cell.font = font_header

    cell.alignment = Alignment(
        horizontal="center",
        vertical="center",
        wrap_text=True
    )


ws.freeze_panes = "E2"
ws.auto_filter.ref = ws.dimensions
ws.row_dimensions[1].height = 48

aplicar_bordes(
    ws,
    1,
    ws.max_row,
    1,
    ws.max_column
)

ajustar_columnas(ws)

ws.column_dimensions["A"].width = 8
ws.column_dimensions["B"].width = 55
ws.column_dimensions["C"].width = 24
ws.column_dimensions["D"].width = 26


# ============================================================
# 7B. UNA PESTAÑA POR UNIDAD FUNCIONAL
# ============================================================

nombres_usados = {"concentrado"}


for cfg in configs_salida:

    uf_key = cfg["key"]

    subset = detalle[
        detalle["UF_KEY"] == uf_key
    ].copy()

    if subset.empty:
        continue


    nombre = nombre_hoja_seguro(
        cfg["nombre_hoja"]
    )

    nombre_base = nombre
    contador = 2

    while nombre.casefold() in nombres_usados:

        sufijo = f" {contador}"

        nombre = (
            nombre_base[
                :31 - len(sufijo)
            ]
            + sufijo
        )

        contador += 1

    nombres_usados.add(nombre.casefold())


    ws_uf = wb.create_sheet(nombre)


    # --------------------------------------------------------
    # Título superior, siguiendo la estructura previa.
    # --------------------------------------------------------

    ws_uf.merge_cells(
        "F1:J1"
    )

    ws_uf["F1"] = "TIPO DE INSTALACION"

    ws_uf["F1"].font = font_bold

    ws_uf["F1"].alignment = Alignment(
        horizontal="center",
        vertical="center"
    )


    # --------------------------------------------------------
    # Encabezados
    # --------------------------------------------------------

    headers_uf = [
        "LOCAL",
        "EQUIPO",
        "CANTIDAD",
        "MARCA",
        "MODELO",
        "INSTALL_A",
        "INSTALL_B",
        "INSTALL_C",
        "M",
        "INSTALL_E",
        "CLASIFICACION"
    ]

    for col, encabezado in enumerate(
        headers_uf,
        start=1
    ):

        cell = ws_uf.cell(
            row=2,
            column=col,
            value=encabezado
        )

        cell.fill = fill_header
        cell.font = font_header

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )


    # --------------------------------------------------------
    # Datos resumidos de la UF
    # --------------------------------------------------------

    subset = subset.sort_values(
        by=[
            "LOCAL",
            "EQUIPO",
            "MARCA",
            "MODELO"
        ],
        key=lambda s:
            s.astype(str).str.upper()
    )


    for _, r in subset.iterrows():

        ws_uf.append([
            r["LOCAL"],
            r["EQUIPO"],
            int(r["CANTIDAD"]),
            r["MARCA"],
            r["MODELO"],

            # Sin Formato.xlsx:
            "",
            "",
            "",
            "",
            "",

            r["CLASIFICACION"]
        ])


    # --------------------------------------------------------
    # Total UF
    # --------------------------------------------------------

    fila_total = (
        ws_uf.max_row + 1
    )

    ws_uf.cell(
        row=fila_total,
        column=2,
        value="TOTAL DE EQUIPOS EN LA UNIDAD FUNCIONAL"
    )

    ws_uf.cell(
        row=fila_total,
        column=3,
        value=int(
            subset["CANTIDAD"].sum()
        )
    )


    for col in range(
        1,
        12
    ):

        cell = ws_uf.cell(
            row=fila_total,
            column=col
        )

        cell.fill = fill_total


    ws_uf.cell(
        row=fila_total,
        column=2
    ).font = font_bold

    ws_uf.cell(
        row=fila_total,
        column=3
    ).font = font_bold


    # --------------------------------------------------------
    # Formato
    # --------------------------------------------------------

    ws_uf.freeze_panes = "A3"

    if fila_total > 3:

        ws_uf.auto_filter.ref = (
            f"A2:K{fila_total - 1}"
        )


    aplicar_bordes(
        ws_uf,
        2,
        fila_total,
        1,
        11
    )


    ws_uf.column_dimensions["A"].width = 42
    ws_uf.column_dimensions["B"].width = 58
    ws_uf.column_dimensions["C"].width = 12
    ws_uf.column_dimensions["D"].width = 24
    ws_uf.column_dimensions["E"].width = 26

    for letra in [
        "F",
        "G",
        "H",
        "I",
        "J"
    ]:
        ws_uf.column_dimensions[
            letra
        ].width = 8

    ws_uf.column_dimensions[
        "K"
    ].width = 18

    ws_uf.row_dimensions[
        2
    ].height = 34


# ------------------------------------------------------------
# Guardar
# ------------------------------------------------------------

wb.save(SALIDA)

print("Excel generado correctamente:")
print(SALIDA)

## 8. Reconcile counts and review results

Reconcile valid source rows against grouped counts. No original performance claim is made.

In [ ]:
# ============================================================
# 8. VALIDACIÓN FINAL
# ============================================================

total_registros = int(
    df["PIEZA"].sum()
)

total_resumen = int(
    detalle["CANTIDAD"].sum()
)


print("============================================================")
print("VALIDACIÓN GENERAL")
print("============================================================")

print(
    "Registros válidos del inventario :",
    total_registros
)

print(
    "Suma de cantidades generadas     :",
    total_resumen
)

print(
    "Coinciden                        :",
    total_registros == total_resumen
)


if total_registros != total_resumen:

    raise ValueError(
        "ERROR: el resumen no coincide "
        "con el número de registros."
    )


# ------------------------------------------------------------
# Resumen general por UF
# ------------------------------------------------------------

resumen_uf = (
    detalle
    .groupby(
        "UF",
        as_index=False
    )["CANTIDAD"]
    .sum()
    .sort_values(
        "CANTIDAD",
        ascending=False
    )
)


print(
    "\nRESUMEN POR UNIDAD FUNCIONAL"
)

display(
    resumen_uf
)


# ------------------------------------------------------------
# Comprobar productos específicos cuando quieras.
# ------------------------------------------------------------

def revisar_producto(texto):

    q = normalizar(texto)

    resultado = detalle[
        detalle["EQUIPO"]
        .apply(normalizar)
        .str.contains(
            q,
            regex=False
        )
    ][
        [
            "UF",
            "LOCAL",
            "EQUIPO",
            "CANTIDAD",
            "MARCA",
            "MODELO"
        ]
    ].copy()


    if resultado.empty:

        print(
            "No se encontraron coincidencias."
        )

    else:

        display(
            resultado.sort_values(
                [
                    "UF",
                    "LOCAL",
                    "EQUIPO"
                ]
            )
        )


# Ejemplo:
# revisar_producto("CARPETA PORTA EXPEDIENTE")

## 9. Download or locate the output

Only the fictional demo workbook should be shared publicly.

In [ ]:
if files is not None:
    files.download(SALIDA)
else:
    print("Output workbook:", os.path.abspath(SALIDA))

## Limitations and verification status

- This is a privacy-adapted V2 source notebook, not a validated release of a V3 useful-life lookup.
- Different classification values within a grouped combination are represented by the most frequent non-empty value, as in V2. Review these cases before operational use.
- Header scanning is limited to the first 20 rows. Ambiguous equally ranked tables are rejected.
- Empty source fields are not inferred or invented.
- Every matching row is counted, including repeated-looking rows; a separate asset-ID policy would be required to deduplicate.
- Source grouping logic was reviewed. This public adaptation has not yet been executed in Python; workbook totals and layout still need runtime validation.

## Expected fictional demo
The generated input has 12 valid rows across three units. Each unit has four assets. These are expected test values, not an executed result.

## Next steps
Run the demo, reconcile the consolidated sheet and each detail sheet, inspect workbook formatting, and add useful-life rules only from an approved non-confidential reference.
